<a href="https://colab.research.google.com/github/ShahidMazumdarAI-hub/VECTOR_STORES-in-Langchain/blob/main/LANGCHAIN_DOC_LOADERS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***VECTOR_STORES in L.CHAIN***

In [ ]:
OPENAI_API_KEY = " " #for security and integrity purposes, didn't share my openai api key
HUGGINGFACEHUB_ACCESS_TOKEN = " " #same goes for huggingface_api_key

In [ ]:
#these vectorstores are helpful in making applications where the requirements are like movie recommendations etc etc, there u mi8 have 10s of lakhs of datas(movies)
#in ur d.base, in case if u want to do smart semantic search of one movie with the other 10s of lakhs of movie, then ur app mi8 turn to be te slowest
#also the plots of these movies must be stored in the d.base in the form of embeddings for which storage mi8 be another issue
#thus all of these can be solved using VECTORSTORES..
#A VECTOR STORE is a system designed 2 store and retrieve data represented as numerical vectors,here u can store the metadata of ur data or the id of ur data etc etc
#here if u want then apne vectors ko in-memory(RAM) ya on-disk(HARD-DRIVE) store kar sakte ho. RAM me aisa hai ki app agar ek baar band kar diya toh datas udd jayenge and HARD DRIVE me even when u off the app,datas rahenge
#u can do similarity search here too
# to enhance FAST SIMILARITY SEARCHES on high dim-vectors, we have something here called as INDEXING(like Clustering where ur datas in d.base is broken down in2 clusters, then the semantic search is conducted)
#u can add new data in ur vectorStore, read them, update the existing datas, remove the datas too etc etc
# Ex-  FAISS ...Facebook ka vectorStore hai yeh, other examples are PINECONE, CHROMA etc etc
# the above examples when used in L.CHAIN use the same methods like from_documents(), from_texts() etc etc etc, add_documents()..kal ko  agar aapko FAISS ke jagah PINECONE use karna hai then the flexibility is allowed, so all methods are the same in all v.stores when used in L.CHAIN
#we will deal with CHROMA now which is a li8 weight open source vector dB

!pip install langchain chromadb openai tiktoken pypdf langchain_openai langchain-community
!pip install langchain langchain_huggingface
from langchain_huggingface import HuggingFaceEmbeddings
!pip install chromadb #we will use the vecor d.Base Chroma
!pip install langchain langchain_community

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma #other v.stores like FAISS, PINECONE etc etc can also be imported here

#lets make a few DOCS here
from langchain_core.documents import Document #document object ki 2 khaasiyat hai, pehla hota hai ki (page_content) where the actual text is stored and 2nd is (meta data) where the meta data which is related to that text is stored

!pip install sentence-transformers

doc1 = Document(
    page_content="Virat kohli hai bhai, most successful captain of the  Indian cricket team, aggressive batting style",
    metadata={"team":"RCB"}
)
doc2 = Document(
    page_content="Dhoni hai bhai, most successful captain of the  Indian cricket team, aggressive batting style",
    metadata={"team":"CSK"}
)
doc3 = Document(
    page_content="JASPRIT hai bhai, most successful captain of the  Indian cricket team, aggressive balling style,good bowler",
    metadata={"team":"MI"}
)
doc4 = Document(
    page_content="Jadeja hai bhai, most successful captain of the  Indian cricket team, aggressive balling style, one of the best bowlers",
    metadata={"team":"CSK"}
)
doc5 = Document(
    page_content="Rohit Sharma hai bhai, most successful captain of the  Indian cricket team, aggressive batting style",
    metadata={"team":"MI"}
)

#now i am making a list where i m storing all of my document objects
docs = [doc1, doc2, doc3, doc4, doc5]

#yaha par u r making ur v.store and making an object of CHROMA and 2 3 chize bataani padti hai yaha, sabse pehle
#apko batana hai ki apka embedding func kya hone wala hai, means jab aap apne doc objects ko apne d.base ke andar insert karogey then uske pehle u need to convert the docs in2 EMBEDDINGS, this can be done by the OpenAIEmbedding model or HUGGINGFACEEMBEDDING model
#dusra ye batana hai ki aap apne jo docs vector ke form me store karna chahte ho, wo kis location pe store hoga?so we will make a folder named as chromadb in our root folder of COLAB n sara embeddings waha pe store hoga
#and lastly we form collections in Chromadb

from langchain_huggingface import HuggingFaceEmbeddings  #since ur OPENAI is no more free now and also openrouter too isnt free, so we are using huggingface

import os
os.environ['HUGGINGFACEHUB_API_TOKEN'] = HUGGINGFACEHUB_ACCESS_TOKEN

vector_store = Chroma(
    embedding_function=HuggingFaceEmbeddings(), #when u want 2 use Open Ai u can type here OpenAIEmbeddings instead of HuggingFaceEmbeddings
    persist_directory='my_chroma_db',
    collection_name='sample'
)
#we will use this vector store to perform diff ops like adding docs
vector_store.add_documents(docs) #this method will add the docs in v.store as well as a unique id is assigned 2 each document so tht baad me using these ids we retrieve the docs

#agar aapko dekhna hai ki aapke Vector Db me kitne docs hai, then u can use get func, isme aap bataa do ki aapko apne docs ke baare me kya ya dekhna hain like..
#mujhe apne docs ke embeddings, docs and metadatas dekhni hai
vector_store.get(include=['embeddings','documents','metadatas'])

#search documents using similarity_search func and u use a query parameter
vector_store.similarity_search(
    query='who among these do batting?',
    k = 2 #matlab aap kitne similar objects ko results me dikhaana chahte ho
)

#in case if u want 2 update an existing  document
updated_doc1 = Document(
     page_content="Virat kohli the former captain of RCB is known for his impressive batting style btw, he drinks black water every morning and is married 2 A.S",
    metadata={"team":"RCB"}

)
vector_store.update_document(document_id='497425d9-e18b-4261-b630-3fe39eafb412', document=updated_doc1)

vector_store.get(include=['embeddings','documents','metadatas']) #viewing the updated doc now

#delete docs
vector_store.delete(ids=['b5c312b7-2b3f-4ce4-81a9-fac20a4df34e'])

vector_store.get(include=['embeddings','documents','metadatas']) #updated, since i deleted a doc in the above cell

